# 1.4 Variables

**Goal:** identify predictors, outcomes, confounders, and data types without mistaking prediction for causation.

**Prerequisites:** earlier Module 1 notebooks. **Runtime:** Python 3.10+, standard library only. Run cells in order; saved outputs are included. Examples are synthetic.

## Contents
1. Variables and their roles
2. Independent variables
3. Dependent variables
4. Confounding variables
5. Continuous, discrete, categorical, and binary variables
6. Feature identification and leakage
7. Practice and solutions

## 1. Variables have both roles and types

A **variable** records a characteristic that may differ between observational units or over time. Its **role** depends on the question; its **type** describes the meaning and possible values. The same field can be a predictor in one study and an outcome in another.

**Worked example:** study hours can predict an exam score. In a different study, a tutoring intervention might predict study hours. “Independent” and “dependent” are therefore contextual roles, not permanent properties of columns.

A useful data dictionary records the variable's name, meaning, unit, semantic type, allowed values, role, and availability time. Include an entity identifier to recognize repeated observations even if it is excluded from the model.

## 2. Independent Variables

An independent variable is an explanatory variable or predictor, commonly written $x$ or $X$. In an experiment, it often means a deliberately manipulated treatment. In observational prediction, it usually means an input feature, whether manipulated or not.

In $\hat y=b+w_1x_1+w_2x_2$, $x_1,x_2$ are input features; $w_1,w_2,b$ are model parameters, not observed predictor columns. Predictors can be correlated with each other. Calling them “independent variables” does **not** assert statistical independence.

**Worked example:** area and number of rooms may both predict sale price and are often correlated. Neither becomes a proven cause of price just because it is used in a regression.

## 3. Dependent Variables

A dependent variable is the outcome, response, label, or target, often $y$ or $Y$. Its type influences the task: a continuous price suggests regression, while a yes/no churn label suggests binary classification. A count target may call for a count model; reducing counts to a binary indicator changes the question.

Distinguish the true/observed outcome $y$ from its prediction $\hat y$. A residual is often $e=y-\hat y$, though conventions vary. Specify how and when labels are measured; a churn label needs a time window and a clear eligibility definition.

**Prediction does not establish causation.** A predictor can be useful because it is a proxy, consequence, or correlate of an outcome. Acting on that predictor may not change the outcome as a predictive association would suggest.

In [1]:
homes = [
    {"area_m2": 50, "rooms": 2, "price": 150000},
    {"area_m2": 80, "rooms": 3, "price": 240000},
    {"area_m2": 100, "rooms": 4, "price": 310000},
]
feature_names = ["area_m2", "rooms"]
X = [[row[name] for name in feature_names] for row in homes]
y = [row["price"] for row in homes]
predictions = [3000 * row["area_m2"] for row in homes]
residuals = [actual-predicted for actual, predicted in zip(y, predictions)]
print("Feature matrix:", X)
print("Target:", y)
print("Toy predictions:", predictions, "residuals:", residuals)

Feature matrix: [[50, 2], [80, 3], [100, 4]]
Target: [150000, 240000, 310000]
Toy predictions: [150000, 240000, 300000] residuals: [0, 0, 10000]


## 4. Confounding Variables

A **confounder** is a variable involved in a causal structure that creates a noncausal association between an exposure and an outcome, often a common cause of both. A simple example is temperature affecting both ice-cream sales and swimming activity, while swimming activity influences drowning risk. Their association does not show that ice cream causes drowning.

Confounding is defined relative to a causal question and assumptions. A variable associated with both exposure and outcome is not automatically a valid adjustment variable. Measurement timing and causal relationships matter.

### Worked example: prior difficulty can reverse an aggregate comparison

Suppose an optional tutoring program is more often taken by students facing a difficult exam. In the synthetic table below, tutoring has a higher pass rate within each difficulty group, but a lower overall pass rate because the tutored group contains more difficult cases. This is an example of **Simpson's paradox**.

The reversal demonstrates how composition can distort an aggregate association. Even the within-group comparison alone does not prove the causal effect of tutoring; unmeasured differences might remain.

In [2]:
cells = [
    {"difficulty": "easy", "tutored": True, "passed": 18, "total": 20},
    {"difficulty": "easy", "tutored": False, "passed": 80, "total": 100},
    {"difficulty": "hard", "tutored": True, "passed": 30, "total": 100},
    {"difficulty": "hard", "tutored": False, "passed": 4, "total": 20},
]
for difficulty in ["easy", "hard"]:
    for tutored in [True, False]:
        cell = next(c for c in cells if c["difficulty"] == difficulty and c["tutored"] == tutored)
        print(difficulty, "tutored=" + str(tutored), "pass rate:", cell["passed"]/cell["total"])
for tutored in [True, False]:
    subset = [c for c in cells if c["tutored"] == tutored]
    rate = sum(c["passed"] for c in subset)/sum(c["total"] for c in subset)
    print("Overall tutored=" + str(tutored), "pass rate:", rate)
# Tutored: 48/120 = 0.40; not tutored: 84/120 = 0.70.
assert 48/120 < 84/120

easy tutored=True pass rate: 0.9
easy tutored=False pass rate: 0.8
hard tutored=True pass rate: 0.3
hard tutored=False pass rate: 0.2
Overall tutored=True pass rate: 0.4
Overall tutored=False pass rate: 0.7


### Addressing confounding thoughtfully

Random assignment of a treatment balances pre-treatment causes **in expectation**, though finite samples can still be imbalanced. In observational studies, design restrictions, stratification, matching, weighting, or regression adjustment may help under specific assumptions. No method guarantees removal of unmeasured confounding.

Not every related variable should be controlled:

- A **mediator** lies on a causal pathway, such as tutoring → study hours → exam score. Adjusting for it changes which effect is estimated.
- A **collider** is a common effect of two variables. Conditioning on it can create a spurious association. For example, selecting only accepted applicants when both test score and an interview affect acceptance can make those two predictors appear related within the selected group.

For introductory work, draw the assumed causal directions and distinguish **prediction** from **effect estimation** before choosing adjustment variables.

## 5. Continuous, Discrete, Categorical, and Binary Variables

| Type | Definition | Example | Typical validation |
|---|---|---|---|
| Continuous | Underlying values span an interval | Duration, height | Units, range, precision |
| Discrete quantitative | Countable numerical values | Number of tickets | Integer/nonnegative if a count |
| Categorical | Group labels, nominal or ordinal | Device OS, risk tier | Allowed levels, missing/unseen levels |
| Binary | Exactly two defined states | Churn yes/no | Explicit coding and reference state |

Binary indicators are commonly categorical but can be treated numerically in appropriate formulas: the mean of a 0/1 event indicator equals its observed event proportion. The code mapping must be stated. A missing binary value is not a third substantive state and should not automatically become zero.

Type and role are independent: a binary variable can be a predictor or a target; a continuous variable can be either as well. Numeric IDs are categorical identifiers even when stored as integers.

In [3]:
churn = [0, 1, 0, 1, 1]
assert set(churn) <= {0, 1}
print("Coding: 1=churned, 0=did not churn")
print("Event count:", sum(churn), "Event proportion:", sum(churn)/len(churn))
feature_dictionary = {
    "tenure_days": {"role": "predictor", "type": "discrete count", "available": "prediction time"},
    "plan": {"role": "predictor", "type": "categorical nominal", "available": "prediction time"},
    "churn_next_30d": {"role": "target", "type": "binary", "available": "after outcome window"},
}
for name, details in feature_dictionary.items():
    print(name, details)

Coding: 1=churned, 0=did not churn
Event count: 3 Event proportion: 0.6
tenure_days {'role': 'predictor', 'type': 'discrete count', 'available': 'prediction time'}
plan {'role': 'predictor', 'type': 'categorical nominal', 'available': 'prediction time'}
churn_next_30d {'role': 'target', 'type': 'binary', 'available': 'after outcome window'}


## 6. Feature identification and target leakage

A valid feature must be available in the intended deployment situation. **Target leakage** occurs when training uses information unavailable at prediction time or derived from the answer. Good apparent performance may then disappear in real use.

**Worked example:** at the start of a subscription month, predict churn over the next 30 days. Current plan and past usage may be available; a cancellation confirmation sent during the next 30 days is not. A feature named “usage” also needs a window: past-week usage can be valid while future-week usage is leakage.

Split data before fitting preprocessing. Scaling or imputation based on all data can transfer held-out information into training. Repeated users may require entity-level splits; future-outcome tasks generally require time-aware evaluation. The right split reflects the intended generalization question.

The following allow-list audit is intentionally explicit: automatic inspection of numeric dtypes cannot determine causal roles or prediction-time availability.

In [4]:
fields = {
    "plan_now": {"available_at_prediction": True, "role": "predictor"},
    "visits_previous_7d": {"available_at_prediction": True, "role": "predictor"},
    "cancellation_email_next_30d": {"available_at_prediction": False, "role": "leakage"},
    "churn_next_30d": {"available_at_prediction": False, "role": "target"},
    "user_id": {"available_at_prediction": True, "role": "identifier"},
}
feature_names = [name for name, info in fields.items()
                 if info["available_at_prediction"] and info["role"] == "predictor"]
print("Approved predictor fields:", feature_names)
print("Target:", "churn_next_30d")
assert "cancellation_email_next_30d" not in feature_names
assert "user_id" not in feature_names

Approved predictor fields: ['plan_now', 'visits_previous_7d']
Target: churn_next_30d


## Practice

1. In a study predicting exam score from study hours and attendance, identify predictors and target.
2. Does calling height and weight “independent variables” assert that height and weight are statistically independent?
3. Explain a plausible confounder of the association between umbrella use and wet streets.
4. Classify age measured precisely, number of visits, product category, and purchase yes/no.
5. A numeric column takes values 0 and 1. What must you learn before interpreting its mean?
6. Why might adjusting for study hours change the interpretation of a tutoring effect?
7. A model predicts loan default using a collection-agency referral flag recorded after default. What is the problem?
8. In the tutoring table, calculate both overall pass rates. Why do they differ from the within-group comparison?
9. Can the same variable be an outcome in one study and a predictor in another?

## Solutions

1. Predictors: study hours and attendance; target: exam score. This is predictive unless a causal design/question is specified.
2. No. It names their input role; the predictors may be strongly associated.
3. Rain can lead both to umbrella use and to wet streets, so umbrellas need not cause wet streets.
4. Continuous; discrete quantitative; categorical nominal; binary categorical, respectively. Completed years of age would be a discretized age representation.
5. Its semantics, which event is coded 1, how missing values are represented, and which units/time period are included. A legitimate 0/1 indicator's mean is its event proportion.
6. If tutoring changes study hours and hours affect scores, hours are a mediator. Adjusting can remove part of the pathway and no longer estimate the total effect.
7. Target leakage: the feature uses post-outcome information unavailable at the intended prediction time.
8. Tutored: $(18+30)/(20+100)=0.40$. Not tutored: $(80+4)/(100+20)=0.70$. Tutored students contain many more hard cases; aggregate composition reverses the within-group pattern.
9. Yes. Study hours might be predicted from an intervention, then used to predict an exam score in a different question. Roles follow the research question.

**Readiness check:** record a variable's role, type, and availability separately. Explain why a predictive feature need not be causal and why adjustment requires causal reasoning.

**Next:** [1.5 Data Collection](5_Data-Collection.ipynb).